### LangChain examples

In [8]:
import os
import getpass
from langchain_openai import ChatOpenAI

In [9]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [10]:
llm = ChatOpenAI(model_name="gpt-5-nano")
llm.invoke("Tell me a joke")

AIMessage(content='Why don’t scientists trust atoms? Because they make up everything.\n\nWant another joke, maybe a different vibe?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 223, 'prompt_tokens': 10, 'total_tokens': 233, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DjjhqtvXy6LqssWR6hEtI58n7Ku4K', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e63eb-4702-7361-9f14-85a5325d7604-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 223, 'total_tokens': 233, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}})

### PromptTemplate and Chain example

In [11]:
from langchain_core.prompts import PromptTemplate

In [12]:
prompt_template = PromptTemplate.from_template("""
You are an experienced copywriter. Write a {num_words} words summary about city Almaty,
using a {tone} tone
""")

chain = prompt_template | llm

In [13]:
response = chain.invoke({ 
    "num_words": 20, 
    "tone": "knowledgeable and engaging"
})

print(response.content)

Almaty blends Kazakh heritage with modern energy, offering lush parks, bustling markets, alpine panoramas, and a vibrant cultural heartbeat today.


### Few shot prompt with LangChain

In [14]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [15]:
examples = [
  {
      "number": 6,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 15,
      "reasoning": "divisible by 5 but not by 7",
      "result": "Abra"
  },
  {
      "number": 12,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 21,
      "reasoning": "divisible by 7 but not by 5",
      "result": "Kadabra"
  },
  {
      "number": 70,
      "reasoning": "divisible by 5 and by 7",
      "result": "Abra Kadabra"
  } ]


example_prompt = PromptTemplate(
    input_variables=["number", "reasoning", "result"],
    template="{number} \\ {reasoning} \\ {result}"
)
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra: {comma_delimited_input_numbers}",
    input_variables=["comma_delimited_input_numbers"]
)

prompt_input = few_shot_prompt.format(comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35.")

In [16]:
response = llm.invoke(prompt_input)
print(response.content)

- 3: None
- 4: None
- 5: Abra
- 7: Kadabra
- 8: None
- 10: Abra
- 11: None
- 13: None
- 35: Abra Kadabra


### Summarizing a document bigger than the LLM’s context window

In [17]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [19]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser

In [20]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)


In [22]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

In [23]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [24]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

In [26]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [27]:
print(summary)

A concise summary: The narrator Ishmael explains that he goes to the sea to cure his malaise and that the sea’s lure, guided by a fate-like force, compels him toward a whaling voyage. He travels from New York to New Bedford, seeking cheap, sea-adjacent lodging, and stays at The Spouter Inn near the docks, where a vast, enigmatic painting and a trove of whaling gear set a grim nautical mood. A rough group of sailors arrives, including a tall, popular harpooneer named Queequeg; the landlord warns that the harpooneer is dangerous but reliably pays. Ishmael is uneasy about sharing a prodigiously large bed with a harpooneer, and the room is sparsely furnished with a sea-chest, a whale-fireboard, a hammock, and a harpoon at the bed’s head. Queequeg enters with a New Zealand head in a bag and performs a quasi-pagan ritual with a small idol, unsettling Ishmael, yet the two end up sharing the bed; Queequeg’s tattooed, cannibalized appearance contrasts with his unexpectedly courteous, civilized 

### Summarizing across documents

In [46]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

In [49]:
wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [50]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [51]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [ ]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [53]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [54]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [55]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': 'Integrated final summary (enhanced with the additional document content)\n\n- General context and setting\n  - Paestum (ancient Poseidonia) is a coastal site in southern Italy’s Cilento, founded circa 600 BCE as a Greek polis on the Tyrrhenian coast. It later came under Lucanian influence and then Roman rule, before declining and being abandoned in the Early Middle Ages. Rediscovered in the 18th century, the site today lies just south of the modern town, with a museum housing Foce del Sele finds. It is accessible by road, the Paestum railway station, and is about 30 km from a nearby airport.\n\n- Greek phase: sacred precincts, urban layout, and early culture\n  - The Greek phase features three Doric temples dating c.530–460 BCE: two adjacent temples to Hera (Juno) and a third to Athena (Minerva). The Temple of Hera II is the best preserved; Hera I retains much of its entablature; the Temple of Athena has notable architrave integrity. The temples were likely painted a

In [57]:
print(full_summary["final_summary"])

Integrated final summary (enhanced with the additional document content)

- General context and setting
  - Paestum (ancient Poseidonia) is a coastal site in southern Italy’s Cilento, founded circa 600 BCE as a Greek polis on the Tyrrhenian coast. It later came under Lucanian influence and then Roman rule, before declining and being abandoned in the Early Middle Ages. Rediscovered in the 18th century, the site today lies just south of the modern town, with a museum housing Foce del Sele finds. It is accessible by road, the Paestum railway station, and is about 30 km from a nearby airport.

- Greek phase: sacred precincts, urban layout, and early culture
  - The Greek phase features three Doric temples dating c.530–460 BCE: two adjacent temples to Hera (Juno) and a third to Athena (Minerva). The Temple of Hera II is the best preserved; Hera I retains much of its entablature; the Temple of Athena has notable architrave integrity. The temples were likely painted and adorned with terracott